Прибери дублікати apply_id


In [1]:
import pandas as pd

In [2]:
df_appli=pd.read_csv('applications(2.0).csv')
df_indust=pd.read_csv('industries(2.0).csv')
print(df_appli)
print(df_appli.size)

                Applied at   Amount  Age   Gender  \
0      11.30.2022 10:26:37  12000.0   29  Чоловік   
1      11.30.2022 10:26:39      NaN   36  Чоловік   
2      11.30.2022 10:26:58   7500.0   34  Чоловік   
3      11.30.2022 10:27:31   1500.0   23    Жінка   
4      11.30.2022 10:27:34   8400.0   33    Жінка   
...                    ...      ...  ...      ...   
13310     01.09.2023 11:01  12000.0   25  Чоловік   
13311     01.09.2023 11:14  10500.0   28  Чоловік   
13312     01.09.2023 11:19   5790.0   25  Чоловік   
13313     01.09.2023 11:28  13500.0   31  Чоловік   
13314     01.09.2023 11:38  12600.0   32  Чоловік   

                           Industry Marital status  External Rating  \
0                        Blockchain          Other              8.0   
1      Public services / Government         Single              3.0   
2              Adtech / Advertising         Single              4.0   
3                           Telecom         Single              0.0   
4       

In [3]:
print(df_indust.head(5))
df_indust.size

                       Industry  Score
0                    Blockchain      0
1  Public services / Government     20
2          Adtech / Advertising     10
3                       Telecom     15
4                    Automotive     15


74

In [4]:
df_appli.columns

Index(['Applied at', 'Amount', 'Age', 'Gender', 'Industry', 'Marital status',
       'External Rating', 'Education level', 'Location', 'applicant_id'],
      dtype='object')

In [5]:
#df_appli["applicant_id"]
df_appli.drop_duplicates(subset='applicant_id', keep='first', inplace=True)
df_appli.size

132780

У полі 'Зовнішній рейтинг' заповни відсутні значення нулями


In [6]:
df_appli['External Rating'].fillna(0, inplace=True)


У полі 'Рівень освіти' заповни відсутнє значення тексту “Середня”


In [7]:
df_appli['Education level'].fillna('Середня', inplace=True)
df_appli.size
df_appli.shape

(13278, 10)

2. Додайте до цього DataFrame дані з файлу industries.csv, а також рейтинги промисловості.

In [8]:
#df_merged=pd.merge(df_appli, df_indust, how='inner', on='Industry')
df_merged= pd.merge(df_appli,df_indust)
df_merged.head(3)
df_merged.shape
#df_merged.columns

(13278, 11)

In [9]:
df_merged.head(2)
#df_merged['Industry'].unique()


,Applied at,Amount,Age,Gender,Industry,Marital status,External Rating,Education level,Location,applicant_id,Score
0,11.30.2022 10:26:37,12000.0,29,Чоловік,Blockchain,Other,8.0,"Вища (бакалавр, спеціаліст, магістр)",Івано-Франківськ чи область,99e7b0dc6cc05dd334d8f38dc26ce9b3,0
1,11.30.2022 10:30:00,NaN,22,Чоловік,Blockchain,Single,2.0,Ще студент вишу,NaN,bef0a5ba4df413cb8e1e3edeaf1f7de3,0


3. Розрахуй рейтинг заявки за наступними умовами:



In [14]:
df_merged['Applied at'] = pd.to_datetime(df_merged['Applied at'], format='mixed', dayfirst=True, errors='coerce')
df_merged['Applied_on_day'] = df_merged['Applied at'].dt.day_name()
df_merged.head(5)
df_merged.

,Applied at,Amount,Age,Gender,Industry,Marital status,External Rating,Education level,Location,applicant_id,Score,Applied_on_day
0,2022-11-30 10:26:37,12000.0,29,Чоловік,Blockchain,Other,8.0,"Вища (бакалавр, спеціаліст, магістр)",Івано-Франківськ чи область,99e7b0dc6cc05dd334d8f38dc26ce9b3,0,Wednesday
1,2022-11-30 10:30:00,NaN,22,Чоловік,Blockchain,Single,2.0,Ще студент вишу,NaN,bef0a5ba4df413cb8e1e3edeaf1f7de3,0,Wednesday
2,2022-11-30 10:36:59,30000.0,29,Чоловік,Blockchain,Single,7.0,"Вища (бакалавр, спеціаліст, магістр)",Київ чи область,d85404bd5576f05b711ad56cad428463,0,Wednesday
3,2022-11-30 10:44:28,16050.0,50,Чоловік,Blockchain,Other,15.0,"Вища (бакалавр, спеціаліст, магістр)",Київ чи область,667563fc9922d71c7c2bfa8e9fba9fbc,0,Wednesday
4,2022-11-30 10:46:30,8550.0,40,Чоловік,Blockchain,Married,1.0,"Вища (бакалавр, спеціаліст, магістр)",Одеса чи область,056d0a80808828a4d4017224fd17491e,0,Wednesday


З чого склали рейтинг:
Якщо вік заявника між 35 та 55, до рейтингу додається 20 балів
Якщо заявника одружений, до рейтингу додається 20 балів
Якщо заявник знаходиться в Києві чи області, до рейтингу додається 10 балів
Значення 'Score' з таблиці industries.csv також додається до заявки (і складається від 0 до 20 балів)
Якщо «Зовнішній рейтинг» більше чи дорівнює 7, до рейтингу додається 20 балів
Якщо «Зовнішній рейтинг» менше чи дорівнює 2, з рейтингу віднімається 20 балів


In [18]:
import numpy as np
age = ((df_merged['Age'] >= 35) & (df_merged['Age'] <= 55)) * 20
day = ~df_merged['Applied_on_day'].isin(['Saturday','Sunday']) * 20
marital = (df_merged['Marital status'] == 'Married') * 20
location = (df_merged['Education level'] == 'Київ чи область') * 10
Industry = np.where(df_merged['Industry'] == 'Score', df_merged['Score'], 0)
positive_rating = (df_merged['External Rating'] >= 7) * 20
negative_rating = (df_merged['External Rating'] <= 2) * -20

In [19]:
df_merged['Total_Score'] = age + day + marital + location + Industry + positive_rating + negative_rating


In [20]:
df_merged.head(2)


,Applied at,Amount,Age,Gender,Industry,Marital status,External Rating,Education level,Location,applicant_id,Score,Applied_on_day,Total_Score
0,2022-11-30 10:26:37,12000.0,29,Чоловік,Blockchain,Other,8.0,"Вища (бакалавр, спеціаліст, магістр)",Івано-Франківськ чи область,99e7b0dc6cc05dd334d8f38dc26ce9b3,0,Wednesday,40
1,2022-11-30 10:30:00,NaN,22,Чоловік,Blockchain,Single,2.0,Ще студент вишу,NaN,bef0a5ba4df413cb8e1e3edeaf1f7de3,0,Wednesday,0


In [21]:


df_merged['Total_Score'] = np.where((df_merged['External Rating'] == 0) | (df_merged['Amount'] == 0), 0, df_merged['Total_Score'])
df_merged

,Applied at,Amount,Age,Gender,Industry,Marital status,External Rating,Education level,Location,applicant_id,Score,Applied_on_day,Total_Score
0,2022-11-30 10:26:37,12000.0,29,Чоловік,Blockchain,Other,8.0,"Вища (бакалавр, спеціаліст, магістр)",Івано-Франківськ чи область,99e7b0dc6cc05dd334d8f38dc26ce9b3,0,Wednesday,40
1,2022-11-30 10:30:00,NaN,22,Чоловік,Blockchain,Single,2.0,Ще студент вишу,NaN,bef0a5ba4df413cb8e1e3edeaf1f7de3,0,Wednesday,0
2,2022-11-30 10:36:59,30000.0,29,Чоловік,Blockchain,Single,7.0,"Вища (бакалавр, спеціаліст, магістр)",Київ чи область,d85404bd5576f05b711ad56cad428463,0,Wednesday,40
3,2022-11-30 10:44:28,16050.0,50,Чоловік,Blockchain,Other,15.0,"Вища (бакалавр, спеціаліст, магістр)",Київ чи область,667563fc9922d71c7c2bfa8e9fba9fbc,0,Wednesday,60
4,2022-11-30 10:46:30,8550.0,40,Чоловік,Blockchain,Married,1.0,"Вища (бакалавр, спеціаліст, магістр)",Одеса чи область,056d0a80808828a4d4017224fd17491e,0,Wednesday,40
...,...,...,...,...,...,...,...,...,...,...,...,...,...
13273,2022-04-12 12:53:00,14403.0,34,Чоловік,Robotics,Married,13.0,"Вища (бакалавр, спеціаліст, магістр)",NaN,3ed663a677fa53827d49322ef876cd7e,10,Tuesday,60
13274,2022-05-12 16:32:00,6000.0,30,Чоловік,Robotics,Married,0.0,"Вища (бакалавр, спеціаліст, магістр)",Львів чи область,825f8a8e84b5c3e810b9d8e3554dafd8,10,Thursday,0
13275,2022-07-12 16:24:00,900.0,18,Чоловік,Robotics,Other,1.0,Ще студент вишу,NaN,5acbd543198ee9c454813d89ee1f670d,10,Tuesday,0
13276,2022-12-14 11:13:46,3900.0,27,Жінка,Robotics,Single,1.0,"Вища (бакалавр, спеціаліст, магістр)",Одеса чи область,cd75c270d6f313368e1f5437de19dfa9,10,Wednesday,0


4. У підсумковій таблиці залишилися лише заявки з рейтингом більше нуля, ці заявки визнані прийнятими.

In [22]:
df_accepted = df_merged[df_merged['Total_Score'] > 0]
df_accepted


,Applied at,Amount,Age,Gender,Industry,Marital status,External Rating,Education level,Location,applicant_id,Score,Applied_on_day,Total_Score
0,2022-11-30 10:26:37,12000.0,29,Чоловік,Blockchain,Other,8.0,"Вища (бакалавр, спеціаліст, магістр)",Івано-Франківськ чи область,99e7b0dc6cc05dd334d8f38dc26ce9b3,0,Wednesday,40
2,2022-11-30 10:36:59,30000.0,29,Чоловік,Blockchain,Single,7.0,"Вища (бакалавр, спеціаліст, магістр)",Київ чи область,d85404bd5576f05b711ad56cad428463,0,Wednesday,40
3,2022-11-30 10:44:28,16050.0,50,Чоловік,Blockchain,Other,15.0,"Вища (бакалавр, спеціаліст, магістр)",Київ чи область,667563fc9922d71c7c2bfa8e9fba9fbc,0,Wednesday,60
4,2022-11-30 10:46:30,8550.0,40,Чоловік,Blockchain,Married,1.0,"Вища (бакалавр, спеціаліст, магістр)",Одеса чи область,056d0a80808828a4d4017224fd17491e,0,Wednesday,40
5,2022-11-30 10:48:36,8400.0,27,Чоловік,Blockchain,Married,6.0,"Вища (бакалавр, спеціаліст, магістр)",Черкаси чи область,56f735f424804136be23a2c0cce2ac65,0,Wednesday,40
...,...,...,...,...,...,...,...,...,...,...,...,...,...
13269,2022-11-30 12:24:29,10260.0,41,Чоловік,Robotics,Married,10.0,"Вища (бакалавр, спеціаліст, магістр)",Київ чи область,74f097d4e71b57de9d1cb7a14637b186,10,Wednesday,80
13270,2022-11-30 19:26:10,10800.0,26,Жінка,Robotics,Single,4.0,"Вища (бакалавр, спеціаліст, магістр)",NaN,392dae911243ecdd585b80451341b93e,10,Wednesday,20
13271,2022-11-30 21:27:10,9600.0,22,Чоловік,Robotics,Married,4.0,"Вища (бакалавр, спеціаліст, магістр)",Хмельницький чи область,bc824c67b10193d15d46e8a49183e71e,10,Wednesday,40
13273,2022-04-12 12:53:00,14403.0,34,Чоловік,Robotics,Married,13.0,"Вища (бакалавр, спеціаліст, магістр)",NaN,3ed663a677fa53827d49322ef876cd7e,10,Tuesday,60


5. Дані з результатами таблиці груп за тижневою подачею заявки, та виведи середній рейтинг прийнятих заявок за кожен тиждень.

In [23]:
df_accepted.loc[:, 'Week'] = df_accepted['Applied at'].dt.to_period('W')
df_weekly_avg = df_accepted.groupby('Week')['Total_Score'].mean().reset_index()
df_weekly_avg

/tmp/ipykernel_294/2468720801.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_accepted.loc[:, 'Week'] = df_accepted['Applied at'].dt.to_period('W')


,Week,Total_Score
0,2022-01-10/2022-01-16,42.503817
1,2022-02-07/2022-02-13,30.769231
2,2022-03-07/2022-03-13,29.230769
3,2022-04-11/2022-04-17,39.008264
4,2022-05-09/2022-05-15,37.441498
5,2022-06-06/2022-06-12,28.230088
6,2022-07-11/2022-07-17,37.319588
7,2022-08-08/2022-08-14,36.700680
8,2022-09-12/2022-09-18,36.923077
9,2022-10-10/2022-10-16,38.909091
